# Exploratory Data Analysis (EDA)
## Project: Agentic AI Powered Explainable Diabetes Prediction & Personalized Health Assistant

This notebook covers the exploratory analysis of the Pima Indians Diabetes Dataset. As a beginner in Machine Learning and Python, it is crucial to understand that before training any model, we must study the data, its structure, distributions, and check for abnormalities (like missing values).

### Objectives:
1. **Load and Inspect**: Load the data and check row count, data types, and statistics.
2. **Zero-value Investigation**: Check for invalid zero values in medical metrics where a zero is physiologically impossible (e.g., Blood Pressure, Glucose, BMI).
3. **Data Visualization**: Create plots to see how different parameters relate to the outcome (Diabetic vs Non-Diabetic).
4. **Preprocessing Foundation**: Formulate a clean-up strategy for training.

In [ ]:
# Import standard data science libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style for clean graphics
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
print("Libraries successfully imported!")

--- 
## Step 1: Load and Inspect the Dataset

In [ ]:
# Load the dataset from the dataset directory
df = pd.read_csv("../dataset/diabetes.csv")

# Display the first 5 rows to see the columns
df.head()

In [ ]:
# Check dataset shape (Rows, Columns) and data types
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
df.info()

In [ ]:
# Get summary statistics for each numerical column
df.describe()

--- 
## Step 2: Handle "Missing" Values (Physiological Zeros)

Looking at `df.describe()`, columns like `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and `BMI` have a minimum value of `0`.
* In medical terms, a person cannot have `0` Blood Pressure, `0` Glucose, or `0` BMI. These are placeholder zeros used instead of `NaN` (Not a Number).
* We will count how many zeros are present in each of these columns.

In [ ]:
# Define columns where 0 is invalid
zero_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

# Count zeros in each column
for col in zero_cols:
    zero_count = (df[col] == 0).sum()
    percentage = (zero_count / len(df)) * 100
    print(f"Column '{col}': {zero_count} zero values ({percentage:.2f}% of data)")

### Strategy to handle invalid zeros:
We shouldn't drop these rows because we would lose too much data (especially for `Insulin` and `SkinThickness` which have nearly 50% and 30% zeros).
Instead, we will replace the zeros with `np.nan` and then **impute** (fill) them with the **median** values of each column based on the target class (`Outcome`). This preserves dataset size and avoids biasing the data.

In [ ]:
# Copy the dataframe to keep original intact
df_clean = df.copy()

# Replace zeros with NaN
for col in zero_cols:
    df_clean[col] = df_clean[col].replace(0, np.nan)

# Impute missing values with column median grouping by Outcome (diabetic vs non-diabetic)
for col in zero_cols:
    median_val_0 = df_clean[df_clean['Outcome'] == 0][col].median()
    median_val_1 = df_clean[df_clean['Outcome'] == 1][col].median()
    
    # Apply class-specific median imputation
    df_clean.loc[(df_clean['Outcome'] == 0) & (df_clean[col].isnull()), col] = median_val_0
    df_clean.loc[(df_clean['Outcome'] == 1) & (df_clean[col].isnull()), col] = median_val_1

# Verify that no nulls or invalid zeros remain in these columns
df_clean[zero_cols].describe()

--- 
## Step 3: Exploratory Data Visualization

In [ ]:
# 1. Visualizing the Target Class Distribution
plt.figure(figsize=(6, 4))
ax = sns.countplot(x="Outcome", data=df_clean, palette="pastel")
plt.title("Distribution of Outcome (0: Non-Diabetic, 1: Diabetic)")
plt.xlabel("Diabetes Outcome")
plt.ylabel("Count")
for p in ax.patches:
    ax.annotate(f'{p.get_height()}', (p.get_x() + 0.35, p.get_height() + 5))
plt.show()

In [ ]:
# 2. Correlation Analysis via Heatmap
plt.figure(figsize=(10, 8))
corr_matrix = df_clean.corr()
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Feature Correlation Heatmap")
plt.show()

### Observation:
* `Glucose` has the highest correlation with `Outcome` (~0.49).
* `BMI` and `Age` also show moderate correlations with `Outcome` (~0.31 and ~0.24).

In [ ]:
# 3. Boxplots: Compare feature values for diabetic vs non-diabetic individuals
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

columns_to_plot = df_clean.columns[:-1]
for i, col in enumerate(columns_to_plot):
    sns.boxplot(x="Outcome", y=col, data=df_clean, ax=axes[i], palette="Set2")
    axes[i].set_title(f"{col} by Outcome")
    axes[i].set_xlabel("Outcome")
    axes[i].set_ylabel(col)

# Hide any unused subplots
for j in range(len(columns_to_plot), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

--- 
## Conclusion of EDA
1. **Imputation works**: Class-specific median imputation fills the missing data naturally without distorting distributions.
2. **Glucose, BMI, and Age** are the strongest univariate features pointing to diabetes diagnosis.
3. **Scaling Required**: The numerical ranges are extremely different (e.g. `DiabetesPedigreeFunction` max is ~2.4, while `Insulin` goes up to 846). Models like Logistic Regression will require scaling to perform optimally.